In [1]:
import pandas as pd

file_path = "/group/moniergrp/dbaral/run_project/input_data"
df = pd.read_csv(file_path +  "/gridmet/Booting_Flowering_GrainFill_1979_2023_v2.csv",
    parse_dates=["date"]
)
planting_harvest_dates = pd.read_csv(file_path + "/planting_harvest/planting_harvest_processed.csv", parse_dates = ["date_planting", "date_harvest"])

In [2]:
def make_stage_file(df, stage_name, suffix):
    if isinstance(stage_name, str):
        mask = df["stage"].eq(stage_name)
    else:
        mask = df["stage"].isin(stage_name)

    out = (
        df.loc[mask]
        .groupby(["county", "year"], as_index=False)
        .agg(
            **{
                f"tmmn_{suffix}": ("tmmn", "mean"),
                f"tmmx_{suffix}": ("tmmx", "mean"),
                f"tmean_{suffix}": ("tmean", "mean"),
            }
        )
        .reset_index(drop=True)
    )
    return out

In [3]:

# Booting / Flowering / Grain fill

booting_df   = make_stage_file(df, "booting", "bo")
flowering_df = make_stage_file(df, ["flowering", "flowering_grainfill"], "fl")
grainfill_df = make_stage_file(df, ["grain_fill", "flowering_grainfill"], "gf")

# Save stage outputs
booting_df.to_csv(file_path + "/booting_feature_variables.csv", index=False)
flowering_df.to_csv(file_path + "/flowering_feature_variables.csv", index=False)
grainfill_df.to_csv(file_path + "/grainfill_feature_variables.csv", index=False)


# #Growing season (May 15 to Sept 15 inclusive)

# df["month"] = df["date"].dt.month
# df["day"] = df["date"].dt.day

# season_df = df[
#     (
#         (df["month"] > 5) | ((df["month"] == 5) & (df["day"] >= 15))
#     ) &
#     (
#         (df["month"] < 9) | ((df["month"] == 9) & (df["day"] <= 15))
#     )
# ]

# growing_season_df = (
#     season_df
#     .groupby(["county", "year"], as_index=False)
#     .agg(
#         tmmn_gs=("tmmn", "mean"),
#         tmmx_gs=("tmmx", "mean"),
#         tmean_gs=("tmean", "mean"),
#     )
#     .sort_values(["county", "year"])
#     .reset_index(drop=True)
# )

# growing_season_df.to_csv(file_path + 
#     "/growing_season_feature_varaibles.csv",
#     index=False
# )



In [4]:
combined_df = df.merge(planting_harvest_dates, on = "year", how = "left")

growing_season_df = combined_df[
    (combined_df["date"] >= combined_df["date_planting"]) &
    (combined_df["date"] <= combined_df["date_harvest"])
]

growing_season_features = (
    growing_season_df
    .groupby(["county", "year"], as_index=False)
    .agg(
        tmmn_gs=("tmmn", "mean"),
        tmmx_gs=("tmmx", "mean"),
        tmean_gs=("tmean", "mean"),
    )
    .sort_values(["county", "year"])
    .reset_index(drop=True)
)

growing_season_features.to_csv(
    file_path + "/gridmet/growing_season_feature_variables.csv",
    index=False
)
